<h1>Sarcasm detection system using custom
Naïve Bayes implementations</h1>

In [7]:
import numpy as np
import pandas as pd
import pandas as pd
import matplotlib.pyplot as plt
import re
import json
import math 
from collections import defaultdict, Counter
from sklearn.metrics  import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [8]:
student_ID = 276690
last_6     = student_ID % 1000000
seed       = last_6 % 100000

In [9]:
df = pd.read_json('/Users/arjun/Downloads/Data/Sarcasm_Headlines_Dataset_v2.json', lines=True)
print(df.shape)
df.head()



(28619, 3)


,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [10]:
#shuffling
df_shuffled = df.sample(frac=1, random_state=seed).reset_index(drop=True)
df_shuffled.head()

,is_sarcastic,headline,article_link
0,1,woman walking alone at night picks up pace aft...,https://local.theonion.com/woman-walking-alone...
1,0,exxon mobil told to hand over decades of clima...,https://www.huffingtonpost.com/entry/exxon-mas...
2,0,top german soccer team hit in explosions,https://www.huffingtonpost.com/entry/borussia-...
3,0,assad's forces take 'capital of revolution',https://www.huffingtonpost.com/entry/assad-tak...
4,1,you to receive 15 pounds of venison sausage fr...,https://www.theonion.com/you-to-receive-15-pou...


In [11]:
#70/15/15 split 
n = len(df_shuffled)
training_upper_lim = int(0.7 * n)
validation_upper_lim = int(0.85 * n)
training_dataset = df_shuffled.iloc[: training_upper_lim]
validation_dataset = df_shuffled.iloc[training_upper_lim:validation_upper_lim]
test_dataset = df_shuffled.iloc[validation_upper_lim:]

In [12]:
print(n)
print(f'train dataset count : {len(training_dataset)}')
print(f'validation dataset count : {len(validation_dataset)}')
print(f'test dataset count : {len(test_dataset)}')
total = len(training_dataset) + len(validation_dataset) + len(test_dataset)
print(f'{'count verified'if total ==n else 'mismatch in count'}')




28619
train dataset count : 20033
validation dataset count : 4293
test dataset count : 4293
count verified


In [13]:
#smoothing
smoothing_sets = {
    0: [0, 0.1, 1],
    1: [0, 0.5, 2],
    2: [0, 1,   5],
}
k = seed % 4 
smoothing_key    = seed % 3
selected_alphas  = smoothing_sets[smoothing_key]

**Task 1 – Tokenisation & Preprocessing**

In [14]:
#basic cleaning
def basic_cleaning (text):
     text = text.lower() #remove stand alone number
     text = re.sub(r"<.*?>","", text) #html
     text = re.sub(r'[?!]*\?[!]+[?!]*|[?!]*![?]+[?!]*', '[INT_MARK]', text)
     text = re.sub(r'!{2,}','[EXC_SEQ]',  text) #Exclamation Sequences: !!, !!!, etc. à[EXC_SEQ]
     text = re.sub(r'\?{2,}','[QUE_SEQ]',  text)  #Question Sequences: ??, ???, etc. à[QUE_SEQ]
     text = re.sub(r'\.{3,}',   '[ELLIP]',    text) #Ellipses: ... à [ELLIP]

     text = re.sub(r"[^\w\s'\[\]]", "", text)  # remove non word/space/apostrophe
     text = re.sub(r"(?<!\w)'|'(?!\w)", '', text)  # remove apostrophes NOT inside words

     text = re.findall(r'\[.*?\]|\w+(?:\'\w+)*', text)
     return text
     

references:
https://www.geeksforgeeks.org/python/how-to-remove-html-tags-from-string-in-python/

https://regex101.com/

https://docs.python.org/3/library/re.html#re.sub #pattern substitution


In [15]:
df_shuffled['headline'].iloc[1]

'exxon mobil told to hand over decades of climate documents in major legal blow'

In [16]:
#testing regex
sample_headlines = [
    "Scientists DISCOVER!!! That Water Is WET!!",   
    "Are? you?? serious????",             
    "Oh reaLly?! come on!!??",      
    "just waiting...",    
    "<b>Breaking news</b>: bomb blast", 
    "senate's new plan to [repeal] but not replace",  
    "a 'damn' letter from a serial killer",    
]

data=[]
for i in sample_headlines:
    word = basic_cleaning(i)
    print(word)




['scientists', 'discover', '[EXC_SEQ]', 'that', 'water', 'is', 'wet', '[EXC_SEQ]']
['are', 'you', '[QUE_SEQ]', 'serious', '[QUE_SEQ]']
['oh', 'really', '[INT_MARK]', 'come', 'on', '[INT_MARK]']
['just', 'waiting', '[ELLIP]']
['breaking', 'news', 'bomb', 'blast']
["senate's", 'new', 'plan', 'to', '[repeal]', 'but', 'not', 'replace']
['a', 'damn', 'letter', 'from', 'a', 'serial', 'killer']


**Step 2 – Negation Handling**

In [17]:
#Negation Handling
negation_words={ "not", "no", "never", "don't", "didn't", "isn't", "wasn't"}
stop_words={"the", "a","also","an","and","are","as","at","be","because","been","but",
            "by","for","from","has","have","however","if","in","is","of",
            "on","or","so","than","that","their","there","these","this","to","was","were"}
vowels = set('aeiou')

def count_vowels(token):
    return sum(1 for ch in token if ch in vowels )
    
def negation_handling(tokens):
    result = []
    i = 0
    while i < len(tokens):
        token = tokens[i]
        if token in negation_words:
            result.append(token)
            n_vowels = count_vowels(token)
            i += 1
            for _ in range(n_vowels):
                if i < len(tokens):
                    result.append("NOT_" + tokens[i])
                    i += 1
        else:
            result.append(token)
            i += 1
    return result
    


In [18]:

def stopword_removal(tokens):
    return [t for t in tokens if t.startswith("NOT_") or t not in stop_words]
 
 
def generate_ngrams(tokens):
    unigrams = tokens
    bigrams  = [f"{tokens[i]}__{tokens[i+1]}" for i in range(len(tokens) - 1)] #double underscore
    return unigrams + bigrams
 

In [19]:
def replace_oov(tokens, vocab):
    return [t if t in vocab else "<UNK>" for t in tokens]

In [20]:
def preprocess(text, vocab=None, ablation_k=None):
    tokens = basic_cleaning(text)

    if ablation_k != 2:
        tokens = negation_handling(tokens)

    if ablation_k != 3:
        tokens = stopword_removal(tokens)

    all_tokens = generate_ngrams(tokens)

    if ablation_k == 0:
        all_tokens = [t for t in all_tokens if "_" in t]
    elif ablation_k == 1:
        all_tokens = [t for t in all_tokens if "_" not in t]

    if vocab is not None:
        all_tokens = replace_oov(all_tokens, vocab)

    return all_tokens

**Step 5 – OOV Handling**

In [21]:
#OOV Handling 

train_tokens = training_dataset["headline"].apply(lambda x: preprocess(x))
vocab            = set(tok for doc in train_tokens for tok in doc)
train_tokens = training_dataset["headline"].apply(lambda x: preprocess(x, vocab))
val_tokens   = validation_dataset["headline"].apply(lambda x: preprocess(x, vocab))
test_tokens  = test_dataset["headline"].apply(lambda x: preprocess(x, vocab))

In [22]:


val_tokens.head()


20033    [two, nonbinary, college, activists, creating,...
20034    [jill, <UNK>, saw, <UNK>, singer's, divorce, c...
20035    [study, major, shift, media, landscape, occurs...
20036    [new, healthier, menu, features, food, wendy's...
20037    [bob, dole, picked, off, large, hawk, circling...
Name: headline, dtype: object

In [23]:
test_tokens.head()

24326    [wildfires, force, colorado, <UNK>, rocky, mou...
24327    [best, chance, defeat, roy, moore, may, democr...
24328                          [pentagon, planning, <UNK>]
24329    [no, NOT_more, adult, conversations, prayers, ...
24330    [alabama, native, channing, tatum, encourages,...
Name: headline, dtype: object

In [24]:
train_tokens.head()

0    [woman, walking, alone, night, picks, up, pace...
1    [exxon, mobil, told, hand, over, decades, clim...
2    [top, german, soccer, team, hit, explosions, t...
3    [assad's, forces, take, capital, revolution, a...
4    [you, receive, 15, pounds, venison, sausage, u...
Name: headline, dtype: object

<h1>Task 2 – Manual Feature Representation</h1>

In [25]:
#Compute Document Frequency
def compute_doc_freq(tokenised_doc):
    doc_freq_count ={}
    for doc in tokenised_doc:
        for tkn in set(doc): #to remove th duplicates
            doc_freq_count[tkn]= doc_freq_count.get(tkn,0) + 1

    return doc_freq_count
doc_freq_count = compute_doc_freq(train_tokens.tolist())

#print(doc_freq_count)

In [26]:
#Rare Word Filtering
#Count Vectoriser
t = (seed % 6) + 2 #this clean up the noise, any word appearing less than t value will be removed

print(f"seed: {seed}")
print(f"seed mod 6: {seed % 6}")
print(f"threshold: {t}")

seed: 76690
seed mod 6: 4
threshold: 6


In [27]:
#sorting in alphabetical order
filtered_tokens = [token for token, count in doc_freq_count.items() if count >= t]
filtered_tokens = sorted(filtered_tokens)
#print(filtered_tokens)

In [28]:
#prime position indexing
def find_prime_number(n):
    prime_list =[True]* (n+1)
    prime_list[0]=False
    prime_list[1]=False
    for i in range(2, int(n**0.5)+1):
        if prime_list[i]:
            for j in range(i*i,n+1,i):
                prime_list[j]=False
    return {i for i in range(2,n+1) if prime_list[i]}
primes_list = find_prime_number(len(filtered_tokens) +10) #buffer added just in case

feature_map = {0: "NON_PRIME"} # colomn 0 is always not prime
for position, token in enumerate(filtered_tokens,start=1):
    if position in primes_list:
        feature_map[position] =token #seperate  colomns for prime number
colomn_order = sorted(feature_map.keys())
prime_positions = set(feature_map.keys()) - {0} #since 0 is already non prime
token_matrix = {tkn: pos for pos,tkn in enumerate(filtered_tokens,start=1)}

In [29]:
print(f"Column 0: {feature_map[0]}")
print(f"Column 2: {feature_map.get(2)}")
print(f"col 0 in prime_positions: {0 in prime_positions}")
print(f"col 2 in prime_positions: {2 in prime_positions}") 

Column 0: NON_PRIME
Column 2: 10
col 0 in prime_positions: False
col 2 in prime_positions: True


In [30]:


#needed some changes in the above code , Vocab size is only 5299/28000 headlines which is very low




In [31]:
def build_count_matrix(tokenised_docs):
    rows=[]
    for doc in tokenised_docs:
        counts = {}
        for token in doc:
            pos=token_matrix.get(token)
            if pos is None:
                continue
            if pos in prime_positions:
                counts[pos]=counts.get(pos,0) +1
            else:
                counts[0] = counts.get(0,0) +1
        rows.append([counts.get(c,0) for c in colomn_order])
    return np.array(rows)


In [32]:
train_data_matrix = build_count_matrix(train_tokens.tolist())
test_data_matrix = build_count_matrix(test_tokens.tolist())
validation_data_matrix = build_count_matrix(val_tokens.tolist())


In [33]:
print(f"train_count: {train_data_matrix.shape}")
print(f"val_count: {test_data_matrix.shape}")
print(f"test_count: {validation_data_matrix.shape}")

train_count: (20033, 771)
val_count: (4293, 771)
test_count: (4293, 771)


**B. TF-IDF**

In [34]:
def compute_tf(doc):
    tf ={}
    total_words = len(doc) if len(doc) > 0 else 1
    for tkn in doc :
        tf[tkn]= tf.get(tkn,0) +1
    for tkn in tf:
        tf[tkn] = tf[tkn]/total_words
    return tf

def compute_idf(df_count, n):
    idf ={}
    for tkn,doc_freq in df_count.items():
        idf[tkn]=math.log((n+1)/(doc_freq+1)) +1
    return idf

num_train_docs = len(training_dataset)
idf = compute_idf(doc_freq_count, num_train_docs)
sample_words = list(idf.items())[:5]
for word, score in sample_words:
    print(f"  '{word}' => IDF = {score:.4f}")

  'alabama__lawmakers' => IDF = 10.2120
  'lawmakers__slowly' => IDF = 10.2120
  'pace' => IDF = 8.7080
  'walking__alone' => IDF = 10.2120
  'alone' => IDF = 7.8607


In [35]:
def compute_tf_idf(tokenised_docs, idf, filtered_tokens):
    tkn_index = {t: i for i, t in enumerate(filtered_tokens)}
    rows = []
    for doc in tokenised_docs:
        vct = np.zeros(len(filtered_tokens))
        tf  = compute_tf(doc)
        for tkn, tf_score in tf.items():
            if tkn in tkn_index:     
                index      = tkn_index[tkn]
                idf_score  = idf.get(tkn, 0)
                vct[index] = tf_score * idf_score
        rows.append(vct)
    return np.array(rows)

# TF-IDF
train_tf_idf = compute_tf_idf(train_tokens.tolist(), idf, doc_freq_count)
val_tf_idf   = compute_tf_idf(val_tokens.tolist(),   idf, doc_freq_count)
test_tf_idf  = compute_tf_idf(test_tokens.tolist(),  idf, doc_freq_count)

print(f"{train_tf_idf.shape}")
print(f"tfidf matrix : {train_tf_idf[0][:10].round(4)}") #first 


(20033, 145956)
tfidf matrix : [0.3294 0.3294 0.2809 0.3294 0.2536 0.2536 0.2585 0.2202 0.3294 0.1407]


**C. Using the training dataset, report the top 10 tokens with the highest Document**

In [36]:
top10 = sorted(doc_freq_count.items(), key=lambda x: -x[1])[:10]
top10_df = pd.DataFrame(top10, columns=['Token', 'Document Frequency'])
print(top10_df.to_string(index=False))


Token  Document Frequency
 with                1366
  new                1143
trump                 996
  man                 941
about                 766
  you                 699
after                 694
  out                 606
   up                 598
   it                 586


<h1> Task 3 — Naive Bayes Variants</h1>

**A. Multinomial NB**

In [37]:
y_train = training_dataset['is_sarcastic'].values
y_val   = validation_dataset['is_sarcastic'].values
y_test  = test_dataset['is_sarcastic'].values

def evaluate(y_true, y_pred, label=''):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1m = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    print(f'{label} Acc={acc:.4f} F1={f1:.4f} F1-sarc={f1m:.4f}')
    return acc, f1, f1m


In [38]:
def train_multinomial_nb(X_train, X_val, y_train, y_val, alphas):
    #Log Priors
    p_non_sarc = np.mean(y_train==0)
    p_sarc = np.mean(y_train == 1)
    
    #log space
    p_non_sarc_log = np.log(p_non_sarc)
    p_sarc_log = np.log(p_sarc)
    
    #Estimate likelihoods
    best_alpha, best_f1 = None, -1 #to track the best model so far
    best_ll_non_sarc, best_ll_sarc = None, None
    
    for alpha in alphas:
        num_features = X_train.shape[1] 
        
        if alpha == 0:
            alpha = 1e-10 
        
        # non sarcasm likelihoods
        non_sarc_matrix = X_train[y_train == 0]
        counts_non_sarc = non_sarc_matrix.sum(axis=0)
        total_non_sarc  = counts_non_sarc.sum()
        ll_non_sarc = np.log( (counts_non_sarc + alpha) / (total_non_sarc + alpha * num_features) )
        
        # sarcasm likelihoods  
        sarc_matrix = X_train[y_train == 1] 
        counts_sarc = sarc_matrix.sum(axis=0)
        total_sarc  = counts_sarc.sum()
        ll_sarc = np.log( (counts_sarc + alpha) / (total_sarc + alpha * num_features) )
        
        # Validation predictions
        scores_non_sarc = p_non_sarc_log + X_val @ ll_non_sarc
        scores_sarc = p_sarc_log + X_val @ ll_sarc 
        y_pred = (scores_sarc > scores_non_sarc).astype(int)
        
        acc, f1, f1m = evaluate(y_val, y_pred, f'alpha={alpha}')
        
        if f1 > best_f1:
            best_f1 = f1
            best_alpha = alpha
            best_ll_non_sarc = ll_non_sarc
            best_ll_sarc = ll_sarc
    
    return best_ll_non_sarc, best_ll_sarc, p_non_sarc_log, p_sarc_log



ll_non_sarc, ll_sarc, p_ns_log, p_s_log = train_multinomial_nb(
    train_data_matrix,      # X_train
    validation_data_matrix, # X_val  
    y_train,                # y_train
    y_val,                  # y_val
    selected_alphas         # alphas
)

# Test
test_scores_non_sarc = p_ns_log + test_data_matrix @ ll_non_sarc
test_scores_sarc = p_s_log + test_data_matrix @ ll_sarc
y_test_pred = (test_scores_sarc > test_scores_non_sarc).astype(int)
evaluate(y_test, y_test_pred, 'Multinomial NB Test')



alpha=1e-10 Acc=0.6266 F1=0.6034 F1-sarc=0.5075
alpha=0.5 Acc=0.6273 F1=0.6040 F1-sarc=0.5080
alpha=2 Acc=0.6280 F1=0.6041 F1-sarc=0.5069
Multinomial NB Test Acc=0.6212 F1=0.5973 F1-sarc=0.4991


(0.6212438853948288, 0.5972907017798916, 0.49907578558225507)

In [39]:
train_binary   = (train_data_matrix > 0).astype(int)
val_binary     = (validation_data_matrix > 0).astype(int)
test_binary    = (test_data_matrix > 0).astype(int)

#log priors
p_non_sarc = np.mean(y_train==0)
p_sarc = np.mean(y_train == 1)
p_non_sarc_log = np.log(p_non_sarc)
p_sarc_log = np.log(p_sarc)

In [40]:
best_alpha_bern, best_f1_bern = None, -1
best_ll_non_sarc_bern, best_ll_sarc_bern = None, None

for alpha in selected_alphas:
    num_features = train_binary.shape[1]
    if alpha == 0:
        alpha = 1e-10 #0 causing error, so assigned a smaller value
    
    non_sarc_matrix = train_binary[y_train == 0] #Non-sarcasm: P(word present | non-sarcastic)
    present_non_sarc = non_sarc_matrix.sum(axis=0)  # #docs containing each word
    docs_non_sarc = non_sarc_matrix.shape[0]
    ll_non_sarc = np.log( (present_non_sarc + alpha) / (docs_non_sarc + 2*alpha) )
    
    # Sarcasm: P(word present | sarcastic)
    sarc_matrix = train_binary[y_train == 1]
    present_sarc = sarc_matrix.sum(axis=0)
    docs_sarc = sarc_matrix.shape[0]
    ll_sarc = np.log( (present_sarc + alpha) / (docs_sarc + 2*alpha) )
    
    #P(word absent) = 1 - P(present)
    ll_absent_non_sarc = np.log(1 - np.exp(ll_non_sarc))
    ll_absent_sarc     = np.log(1 - np.exp(ll_sarc))
    
    # Validation: present*ll_present + absent*ll_absent
    scores_non_sarc = (p_non_sarc_log + 
                      val_binary @ ll_non_sarc + 
                      (1-val_binary) @ ll_absent_non_sarc)
    
    scores_sarc = (p_sarc_log + 
                  val_binary @ ll_sarc + 
                  (1-val_binary) @ ll_absent_sarc)
    
    y_pred = (scores_sarc > scores_non_sarc).astype(int)
    
    acc, f1, f1m = evaluate(y_val, y_pred, f'alpha={alpha}')
    
    if f1 > best_f1_bern:
        best_f1_bern = f1
        best_alpha_bern = alpha
        best_ll_non_sarc_bern = (ll_non_sarc, ll_absent_non_sarc)
        best_ll_sarc_bern = (ll_sarc, ll_absent_sarc)

alpha=1e-10 Acc=0.6289 F1=0.6064 F1-sarc=0.5121
alpha=0.5 Acc=0.6294 F1=0.6068 F1-sarc=0.5124
alpha=2 Acc=0.6296 F1=0.6063 F1-sarc=0.5105


In [41]:
#testing best model
(ll_non_bern, ll_absent_non_bern) = best_ll_non_sarc_bern
(ll_sarc_bern, ll_absent_sarc_bern) = best_ll_sarc_bern

test_scores_non_sarc = (p_non_sarc_log + test_binary @ ll_non_bern + (1-test_binary) @ ll_absent_non_bern)
test_scores_sarc = (p_sarc_log + test_binary @ ll_sarc_bern + (1-test_binary) @ ll_absent_sarc_bern)
y_test_pred_bern = (test_scores_sarc > test_scores_non_sarc).astype(int)
evaluate(y_test, y_test_pred_bern, 'Bernoulli NB Test')

Bernoulli NB Test Acc=0.6210 F1=0.5983 F1-sarc=0.5029


(0.6210109480549733, 0.5983362670088053, 0.5029025358997862)

<h1>Numeric Feature Engineering</h1>

<h1>Task 4 – Numeric Meta-Features & Imputation</h1>

In [42]:
def extract_num_features(df):
    num_features = pd.DataFrame(index=df.index)
    
    num_features['word_count'] = df['headline'].str.split().str.len()
    num_features['char_len'] = df['headline'].str.len()
    num_features['upper_count'] = df['headline'].str.count(r'\b[A-Z]+\b')
    num_features['excl_count'] = df['headline'].str.count('!')
    num_features['ques_count'] = df['headline'].str.count(r'\?')
    num_features['neg_count'] = df['headline'].str.lower().str.count('|'.join(negation_words))
    num_features['sarc_ratio'] = np.where(
        num_features['word_count'] > 0,
        num_features['upper_count'] / num_features['word_count'],
        0.5) # Sarcasm ratio
    
    return num_features
    

train_features = extract_num_features(training_dataset)
val_features = extract_num_features(validation_dataset)
test_features = extract_num_features(test_dataset)

print("Features shape:", train_features.shape)
print(train_features.head())

Features shape: (20033, 7)
   word_count  char_len  upper_count  excl_count  ques_count  neg_count  \
0          18       110            0           0           0          0   
1          14        78            0           0           0          0   
2           7        40            0           0           0          0   
3           6        43            0           0           0          0   
4          10        54            0           0           0          0   

   sarc_ratio  
0         0.0  
1         0.0  
2         0.0  
3         0.0  
4         0.0  


**A. Inject Missing Values**

In [43]:
def inject_missing(df, missing_rate=0.05):
    
    n_missing = int(len(df) * len(df.columns) * missing_rate)
    flat_indices = np.random.choice(
        len(df) * len(df.columns), 
        n_missing, 
        replace=False
    )
    
    rows = flat_indices // len(df.columns)
    cols = flat_indices % len(df.columns)
    
    df.iloc[0, 6] = np.nan
    
    for r, c in zip(rows, cols):
        df.iloc[r, c] = np.nan
        
    return df

train_features_miss = inject_missing(train_features.copy())
val_features_miss = inject_missing(val_features.copy())
test_features_miss = inject_missing(test_features.copy())

**B. Handle Missing Values**

In [44]:
#mean imputation
train_mean = train_features_miss.mean()
train_mean_imp = train_features_miss.fillna(train_mean)
val_mean_imp = val_features_miss.fillna(train_mean)


ll_ns_mean, ll_s_mean, p_ns_log_mean, p_s_log_mean = train_multinomial_nb(
    train_mean_imp, val_mean_imp, y_train, y_val, selected_alphas
)

#test predictions
test_mean_imp = test_features_miss.fillna(train_mean)
test_scores_ns_mean = p_ns_log_mean + test_mean_imp @ ll_ns_mean
test_scores_s_mean = p_s_log_mean + test_mean_imp @ ll_s_mean
y_test_pred_mean = (test_scores_s_mean > test_scores_ns_mean).astype(int)

evaluate(y_test, y_test_pred_mean, 'Global Mean Imputation')

#class-conditional mean imputation
train_mean_non_sarc = train_features_miss[y_train == 0].mean()
train_mean_sarc = train_features_miss[y_train == 1].mean()

class_mean_imp = train_features_miss.copy()
class_mean_imp.loc[y_train == 0] = class_mean_imp.loc[y_train == 0].fillna(train_mean_non_sarc)
class_mean_imp.loc[y_train == 1] = class_mean_imp.loc[y_train == 1].fillna(train_mean_sarc)

val_class_imp = val_features_miss.fillna(train_mean) 


ll_ns_class, ll_s_class, p_ns_log_class, p_s_log_class = train_multinomial_nb(
    class_mean_imp, val_class_imp, y_train, y_val, selected_alphas
)

# Test predictions
test_class_imp = test_features_miss.fillna(train_mean)
test_scores_ns_class = p_ns_log_class + test_class_imp @ ll_ns_class
test_scores_s_class = p_s_log_class + test_class_imp @ ll_s_class
y_test_pred_class = (test_scores_s_class > test_scores_ns_class).astype(int)

evaluate(y_test, y_test_pred_class, 'Class Mean Imputation')


alpha=1e-10 Acc=0.5444 F1=0.4836 F1-sarc=0.3064
alpha=0.5 Acc=0.5444 F1=0.4836 F1-sarc=0.3064
alpha=2 Acc=0.5444 F1=0.4836 F1-sarc=0.3064
Global Mean Imputation Acc=0.5551 F1=0.4997 F1-sarc=0.3331
alpha=1e-10 Acc=0.5474 F1=0.4927 F1-sarc=0.3260
alpha=0.5 Acc=0.5476 F1=0.4927 F1-sarc=0.3257
alpha=2 Acc=0.5476 F1=0.4927 F1-sarc=0.3257
Class Mean Imputation Acc=0.5581 F1=0.5074 F1-sarc=0.3492


(0.5581178662939669, 0.5073596127025857, 0.34922813036020584)

In [45]:
#write comparison code here

<h1>Task 5 – Outlier Detection</h1>

**A. Detect Outliers**

In [48]:
def detect_outliers_iqr(X, factor=1.5):
    Q1 = X.quantile(0.25)
    Q3 = X.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR
    outliers = ((X < lower_bound)|(X > upper_bound)).any(axis=1) 
    
    print(f"Outliers: {outliers.sum()} out of {len(X)} ,percentage: ({outliers.mean()*100:.1f}%)")
    return outliers, lower_bound, upper_bound

outlier_train, lower_bound, upper_bound = detect_outliers_iqr(train_mean_imp) #class mean imputed data


Outliers: 6382 out of 20033 ,percentage: (31.9%)


**B. Treat Outliers**

In [58]:
print("\nOutliers by class:")
outlier_labels = pd.Series(y_train[outlier_train])
print(outlier_labels.value_counts(normalize=True))

# Clamp values to boundary
X_train_clamped = class_mean_imp.clip(lower=lower_bound, upper=upper_bound, axis=1)
X_val_clamped = val_class_imp.clip(lower=upper_bound, upper=upper_bound, axis=1)
X_test_clamped = test_class_imp.clip(lower=lower_bound, upper=upper_bound, axis=1)



Outliers by class:
0    0.521467
1    0.478533
Name: proportion, dtype: float64


In [65]:
#non sarcastic 0
#sarcastic 1

**performace report and class distribution after clamping**

In [66]:
ll_ns_base, ll_s_base, p_ns_log_base, p_s_log_base = train_multinomial_nb(
    class_mean_imp, val_class_imp, y_train, y_val, selected_alphas
)

ll_ns_clamp, ll_s_clamp, p_ns_log_clamp, p_s_log_clamp = train_multinomial_nb(
    X_train_clamped, X_val_clamped, y_train, y_val, selected_alphas
)

# Predictions
test_scores_ns_base = p_ns_log_base + test_class_imp @ ll_ns_base
test_scores_s_base = p_s_log_base + test_class_imp @ ll_s_base
y_test_base = (test_scores_s_base > test_scores_ns_base).astype(int)

test_scores_ns_clamp = p_ns_log_clamp + X_test_clamped @ ll_ns_clamp
test_scores_s_clamp = p_s_log_clamp + X_test_clamped @ ll_s_clamp
y_test_clamp = (test_scores_s_clamp > test_scores_ns_clamp).astype(int)

evaluate(y_test, y_test_base, 'Baseline (w/ outliers)')
evaluate(y_test, y_test_clamp, 'IQR Clamping')

alpha=1e-10 Acc=0.5474 F1=0.4927 F1-sarc=0.3260
alpha=0.5 Acc=0.5476 F1=0.4927 F1-sarc=0.3257
alpha=2 Acc=0.5476 F1=0.4927 F1-sarc=0.3257
alpha=1e-10 Acc=0.5157 F1=0.3402 F1-sarc=0.0000
alpha=0.5 Acc=0.5157 F1=0.3402 F1-sarc=0.0000
alpha=2 Acc=0.5157 F1=0.3402 F1-sarc=0.0000
Baseline (w/ outliers) Acc=0.5581 F1=0.5074 F1-sarc=0.3492
IQR Clamping Acc=0.5292 F1=0.3933 F1-sarc=0.1061


(0.5292336361518751, 0.39331101527359436, 0.10614772224679346)

<h1>Task 6 – Scaling Study</h1>

In [98]:
X_train_scale = X_train_clamped.values.copy()
X_val_scale = X_val_clamped.values.copy()  
X_test_scale = X_test_clamped.values.copy()

In [100]:
def train_gaussian_nb(X_train, X_val, y_train, y_val):
    p_ns_log = np.log(np.mean(y_train==0)); p_s_log = np.log(np.mean(y_train==1))
    ll_ns = np.mean(X_train[y_train==0],0)      # (7,) not [None,:]
    ll_s = np.mean(X_train[y_train==1],0)       # (7,)
    return ll_ns, ll_s, p_ns_log, p_s_log

In [101]:
# Raw
ll_ns_r,ll_s_r,p_ns_r,p_s_r = train_gaussian_nb(X_train_scale,X_val_scale,y_train,y_val)
y_test_r = (p_s_r + X_test_scale@ll_s_r > p_ns_r + X_test_scale@ll_ns_r).astype(int)
evaluate(y_test, y_test_r, 'Raw Clamped')

Raw Clamped Acc=0.4845 F1=0.3264 F1-sarc=0.6528


(0.484509666899604, 0.32637690255766516, 0.6527538051153303)

In [102]:
# MinMax  
X_train_m,_,X_test_m = minmax_scale(X_train_scale,X_val_scale,X_test_scale)
ll_ns_m,ll_s_m,p_ns_m,p_s_m = train_gaussian_nb(X_train_m,X_val_scale,y_train,y_val)
y_test_m = (p_s_m + X_test_m@ll_s_m > p_ns_m + X_test_m@ll_ns_m).astype(int)
evaluate(y_test, y_test_m, 'MinMax')


MinMax Acc=0.5155 F1=0.3401 F1-sarc=0.0000


(0.515490333100396, 0.34014755610205966, 0.0)

In [103]:
# Standard
X_train_s,_,X_test_s = standardize(X_train_scale,X_val_scale,X_test_scale)
ll_ns_s,ll_s_s,p_ns_s,p_s_s = train_gaussian_nb(X_train_s,X_val_scale,y_train,y_val)
y_test_s = (p_s_s + X_test_s@ll_s_s > p_ns_s + X_test_s@ll_ns_s).astype(int)
evaluate(y_test, y_test_s, 'Standardized')


Standardized Acc=0.5523 F1=0.5436 F1-sarc=0.4808


(0.5522944327975775, 0.5436456421059338, 0.4808211777417612)

In [104]:
comparison_data = {
    'Method': ['Raw Clamped', 'MinMax', 'Standardized'],
    'Accuracy': [f"{acc_raw:.4f}", f"{acc_mm:.4f}", f"{acc_std:.4f}"],
    'F1-macro': [f"{f1_raw:.4f}", f"{f1_mm:.4f}", f"{f1_std:.4f}"],
    'F1-sarc': [f"{f1m_raw:.4f}", f"{f1m_mm:.4f}", f"{f1m_std:.4f}"]
}

comp_df = pd.DataFrame(comparison_data)
print(comp_df.to_string(index=False))
print("\nBest: Standardised (Gaussian assumes normal dist ✓)")


      Method Accuracy F1-macro F1-sarc
 Raw Clamped   0.5292   0.3933  0.1061
      MinMax   0.5155   0.3401  0.0000
Standardized   0.5155   0.3401  0.0000

Best: Standardised (Gaussian assumes normal dist ✓)
